# C7-cnn-transfer — Practice p07 — Solution

The NumPy routine is the component-form reference. Stacking the two kernels
on axis 0 and inserting the single input-channel axis produces the required
`(2, 1, 3, 3)` torch weight tensor.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention (no pretrained weights here)
SEED = 20260804
torch.manual_seed(SEED)

rng = np.random.default_rng(SEED)
img = np.zeros((16, 16))
img[4, :] = 1.0
img[:, 10] = 1.0
img += 0.1 * rng.standard_normal((16, 16))

def conv2d_valid(image, kernel):
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    kh, kw = kernel.shape
    out = np.zeros((image.shape[0] - kh + 1, image.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i + kh, j:j + kw] * kernel).sum()
    return out

K_spot = np.array([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]])
K_blur = np.ones((3, 3)) / 9.0
maps_np = np.stack((conv2d_valid(img, K_spot), conv2d_valid(img, K_blur)))

bank = nn.Conv2d(1, 2, kernel_size=3, bias=False)
bank.weight = nn.Parameter(
    torch.as_tensor(np.stack((K_spot, K_blur))).reshape(2, 1, 3, 3),
    requires_grad=False,
)
with torch.inference_mode():
    maps_t = bank(torch.as_tensor(img).reshape(1, 1, 16, 16))

gap = float(np.abs(maps_t.squeeze(0).numpy() - maps_np).max())
shapes_ok = (maps_np.shape == (2, 14, 14) and tuple(maps_t.shape) == (1, 2, 14, 14))


### Answer check

In [ ]:
assert tuple(bank.weight.shape) == (2, 1, 3, 3)
assert bank.weight.requires_grad is False
assert shapes_ok
assert gap < 1e-12
